In [1]:
arm_path = "ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/armenia_base_consolidada_reducida.csv.gz"
per_path = "ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/pereira_base_consolidada_reducida.csv.gz"
cali_path = "ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/cali_base_consolidada_reducida.csv.gz"


Crear datasets

In [2]:
import pandas as pd

arm  = pd.read_csv(arm_path, compression="gzip", low_memory=False)
per  = pd.read_csv(per_path, compression="gzip", low_memory=False)
cali = pd.read_csv(cali_path, compression="gzip", low_memory=False)

print("Armenia:", arm.shape)
print("Pereira:", per.shape)
print("Cali:", cali.shape)

Armenia: (275641, 35)
Pereira: (409670, 35)
Cali: (1822869, 35)


Validar y alinear columnas

In [3]:
common_cols = sorted(set(arm.columns) & set(per.columns) & set(cali.columns))
print("Columnas comunes:", len(common_cols))

arm2  = arm[common_cols]
per2  = per[common_cols]
cali2 = cali[common_cols]

Columnas comunes: 35


In [4]:
df_all = pd.concat([arm2, per2, cali2], ignore_index=True)
print("Base consolidada (3 ciudades):", df_all.shape)


Base consolidada (3 ciudades): (2508180, 35)


In [5]:
# Este bloque revisa duplicados en cada dataset por separado.
# Usa:
# - Llave básica: identifica persona dentro de vivienda/hogar.
# - Llave mejorada: añade UA_CLASE (cuando exista) para diferenciar mejor.

def audit_duplicates(df, nombre):
    print(f"\n===== {nombre} =====")
    print("Filas totales:", df.shape[0])
    print("Columnas:", df.shape[1])

    # Definimos llaves solo con columnas que realmente existan en el dataset
    key_basic_candidates = ["U_DPTO","U_MPIO","U_VIVIENDA","P_NROHOG","P_NRO_PER"]
    key_best_candidates  = ["U_DPTO","U_MPIO","UA_CLASE","U_VIVIENDA","P_NROHOG","P_NRO_PER"]

    key_basic = [c for c in key_basic_candidates if c in df.columns]
    key_best  = [c for c in key_best_candidates  if c in df.columns]

    print("Llave BÁSICA usada:", key_basic)
    print("Llave MEJORADA usada:", key_best)

    if len(key_basic) > 0:
        dup_basic = df.duplicated(subset=key_basic).sum()
        uniq_basic = df.drop_duplicates(subset=key_basic).shape[0]
        print("Duplicados (básica):", dup_basic)
        print("Únicos     (básica):", uniq_basic)

    if len(key_best) > 0:
        dup_best = df.duplicated(subset=key_best).sum()
        uniq_best = df.drop_duplicates(subset=key_best).shape[0]
        print("Duplicados (mejorada):", dup_best)
        print("Únicos     (mejorada):", uniq_best)

        # Si aún hay duplicados con la llave mejorada, mostramos ejemplos
        if dup_best > 0:
            print("Ejemplos de duplicados (primeras 10 filas):")
            dups = df[df.duplicated(subset=key_best, keep=False)]
            display(dups.head(10))

# Ejecutar auditoría para cada ciudad
audit_duplicates(arm,  "Armenia")
audit_duplicates(per,  "Pereira")
audit_duplicates(cali, "Cali")



===== Armenia =====
Filas totales: 275641
Columnas: 35
Llave BÁSICA usada: ['U_DPTO', 'U_MPIO', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Llave MEJORADA usada: ['U_DPTO', 'U_MPIO', 'UA_CLASE', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Duplicados (básica): 273809
Únicos     (básica): 1832
Duplicados (mejorada): 273511
Únicos     (mejorada): 2130
Ejemplos de duplicados (primeras 10 filas):


,COD_DANE_ANM,COD_ENCUESTAS,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,...,VA1_ESTRATO,VA_EE,VB_ACU,VC_ALC,VD_GAS,VE_RECBAS,VF_INTERNET,V_MAT_PARED,V_MAT_PISO,V_TIPO_VIV
0,6300110000000000010115,594972,4.0,3.0,2.0,1.0,2.0,1.0,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
1,6300110000000000010115,594972,4.0,3.0,2.0,NaN,2.0,NaN,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
2,6300110000000000010115,594972,4.0,3.0,2.0,NaN,1.0,NaN,1.0,6,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
3,6300110000000000010115,594972,4.0,3.0,2.0,NaN,1.0,NaN,1.0,3,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
4,6300110000000000010115,594988,5.0,3.0,2.0,NaN,2.0,NaN,1.0,7,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0
5,6300110000000000010115,594988,5.0,3.0,2.0,NaN,2.0,NaN,1.0,6,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0
6,6300110000000000010115,594988,5.0,3.0,2.0,NaN,1.0,NaN,1.0,2,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0
7,6300110000000000010115,594988,5.0,3.0,2.0,NaN,2.0,NaN,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0
8,6300110000000000010115,594988,5.0,3.0,2.0,NaN,2.0,NaN,1.0,12,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0
9,6300110000000000010115,595002,2.0,3.0,2.0,NaN,2.0,NaN,1.0,11,...,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0



===== Pereira =====
Filas totales: 409670
Columnas: 35
Llave BÁSICA usada: ['U_DPTO', 'U_MPIO', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Llave MEJORADA usada: ['U_DPTO', 'U_MPIO', 'UA_CLASE', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Duplicados (básica): 406431
Únicos     (básica): 3239
Duplicados (mejorada): 405748
Únicos     (mejorada): 3922
Ejemplos de duplicados (primeras 10 filas):


,COD_DANE_ANM,COD_ENCUESTAS,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,...,VA1_ESTRATO,VA_EE,VB_ACU,VC_ALC,VD_GAS,VE_RECBAS,VF_INTERNET,V_MAT_PARED,V_MAT_PISO,V_TIPO_VIV
0,6600110000000007010802,332764,4.0,5.0,3.0,NaN,2.0,NaN,1.0,9,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,6600110000000007010802,332764,4.0,5.0,3.0,NaN,1.0,NaN,1.0,2,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,6600110000000007010802,332764,4.0,5.0,3.0,NaN,2.0,NaN,1.0,6,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,6600110000000007010802,332764,4.0,5.0,3.0,NaN,NaN,NaN,NaN,1,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,6600110000000004010211,333127,2.0,5.0,2.0,NaN,2.0,NaN,1.0,14,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
5,6600110000000004010211,333127,2.0,5.0,2.0,NaN,1.0,NaN,1.0,5,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
6,6600110000000003020705,380879,3.0,5.0,3.0,NaN,2.0,NaN,1.0,16,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
7,6600110000000003020705,380879,3.0,5.0,3.0,NaN,2.0,NaN,1.0,12,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
8,6600110000000003020705,380879,3.0,5.0,3.0,NaN,1.0,NaN,1.0,7,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
9,6600110000000003020705,380893,2.0,5.0,3.0,NaN,2.0,NaN,1.0,15,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0



===== Cali =====
Filas totales: 1822869
Columnas: 35
Llave BÁSICA usada: ['U_DPTO', 'U_MPIO', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Llave MEJORADA usada: ['U_DPTO', 'U_MPIO', 'UA_CLASE', 'U_VIVIENDA', 'P_NROHOG', 'P_NRO_PER']
Duplicados (básica): 1813434
Únicos     (básica): 9435
Duplicados (mejorada): 1812860
Únicos     (mejorada): 10009
Ejemplos de duplicados (primeras 10 filas):


,COD_DANE_ANM,COD_ENCUESTAS,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,...,VA1_ESTRATO,VA_EE,VB_ACU,VC_ALC,VD_GAS,VE_RECBAS,VF_INTERNET,V_MAT_PARED,V_MAT_PISO,V_TIPO_VIV
0,7600110000000002210101,335034,2.0,5.0,2.0,NaN,2.0,NaN,1.0,17,...,4.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0
1,7600110000000002210101,335034,2.0,5.0,2.0,NaN,2.0,NaN,1.0,19,...,4.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0
2,7600110000000002210106,335036,7.0,7.0,4.0,NaN,1.0,NaN,1.0,2,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
3,7600110000000002210106,335036,7.0,7.0,4.0,NaN,2.0,NaN,1.0,5,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
4,7600110000000002210106,335036,7.0,7.0,4.0,NaN,2.0,NaN,1.0,19,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
5,7600110000000002210106,335036,7.0,7.0,4.0,NaN,2.0,NaN,1.0,11,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
6,7600110000000002210106,335036,7.0,7.0,4.0,3.0,2.0,1.0,1.0,17,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
7,7600110000000002210106,335036,7.0,7.0,4.0,NaN,2.0,NaN,1.0,11,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
8,7600110000000002210106,335036,7.0,7.0,4.0,NaN,2.0,NaN,1.0,13,...,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0
9,7600110000000002130217,335037,2.0,5.0,4.0,3.0,2.0,1.0,1.0,13,...,5.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0


Imprimir

In [6]:
out_path = "ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/desigualdad_3ciudades.csv.gz"

df_all.to_csv(out_path, index=False, compression="gzip")

print("Archivo guardado en:", out_path)

Archivo guardado en: ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/desigualdad_3ciudades.csv.gz


In [2]:
import pandas as pd
from pathlib import Path

# Ruta
file_path = Path("ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/desigualdad_3ciudades.csv.gz")

# Leer archivo
df = pd.read_csv(file_path, compression="gzip", low_memory=False)

# Ver columnas
print("Columnas:")
print(df.columns.tolist())

# Vista rápida
print("\nPrimeras filas:")
display(df.head(3))
file_path = Path("ETAPA 1 - CONSTRUCCIÓN BASE DE DATOS/outputs/desigualdad_3ciudades.csv.gz")

# Leer archivo
df = pd.read_csv(file_path, compression="gzip", low_memory=False)

# Ver columnas
print("Columnas:")
print(df.columns.tolist())

# Vista rápida
print("\nPrimeras filas:")
display(df.head(3))

Columnas:
['COD_DANE_ANM', 'COD_ENCUESTAS', 'HA_TOT_PER', 'H_NRO_CUARTOS', 'H_NRO_DORMIT', 'PA1_CALIDAD_SERV', 'PA_ASISTENCIA', 'PA_LO_ATENDIERON', 'P_ALFABETA', 'P_EDADR', 'P_ENFERMO', 'P_NIVEL_ANOSR', 'P_NROHOG', 'P_NRO_PER', 'P_PARENTESCOR', 'P_SEXO', 'P_TRABAJO', 'UA1_LOCALIDAD', 'UA_CLASE', 'U_DPTO', 'U_MPIO', 'U_MZA', 'U_SECC_URB', 'U_SECT_URB', 'U_VIVIENDA', 'VA1_ESTRATO', 'VA_EE', 'VB_ACU', 'VC_ALC', 'VD_GAS', 'VE_RECBAS', 'VF_INTERNET', 'V_MAT_PARED', 'V_MAT_PISO', 'V_TIPO_VIV']

Primeras filas:


,COD_DANE_ANM,COD_ENCUESTAS,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,...,VA1_ESTRATO,VA_EE,VB_ACU,VC_ALC,VD_GAS,VE_RECBAS,VF_INTERNET,V_MAT_PARED,V_MAT_PISO,V_TIPO_VIV
0,6300110000000000010115,594972,4.0,3.0,2.0,1.0,2.0,1.0,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
1,6300110000000000010115,594972,4.0,3.0,2.0,NaN,2.0,NaN,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
2,6300110000000000010115,594972,4.0,3.0,2.0,NaN,1.0,NaN,1.0,6,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0


Columnas:
['COD_DANE_ANM', 'COD_ENCUESTAS', 'HA_TOT_PER', 'H_NRO_CUARTOS', 'H_NRO_DORMIT', 'PA1_CALIDAD_SERV', 'PA_ASISTENCIA', 'PA_LO_ATENDIERON', 'P_ALFABETA', 'P_EDADR', 'P_ENFERMO', 'P_NIVEL_ANOSR', 'P_NROHOG', 'P_NRO_PER', 'P_PARENTESCOR', 'P_SEXO', 'P_TRABAJO', 'UA1_LOCALIDAD', 'UA_CLASE', 'U_DPTO', 'U_MPIO', 'U_MZA', 'U_SECC_URB', 'U_SECT_URB', 'U_VIVIENDA', 'VA1_ESTRATO', 'VA_EE', 'VB_ACU', 'VC_ALC', 'VD_GAS', 'VE_RECBAS', 'VF_INTERNET', 'V_MAT_PARED', 'V_MAT_PISO', 'V_TIPO_VIV']

Primeras filas:


,COD_DANE_ANM,COD_ENCUESTAS,HA_TOT_PER,H_NRO_CUARTOS,H_NRO_DORMIT,PA1_CALIDAD_SERV,PA_ASISTENCIA,PA_LO_ATENDIERON,P_ALFABETA,P_EDADR,...,VA1_ESTRATO,VA_EE,VB_ACU,VC_ALC,VD_GAS,VE_RECBAS,VF_INTERNET,V_MAT_PARED,V_MAT_PISO,V_TIPO_VIV
0,6300110000000000010115,594972,4.0,3.0,2.0,1.0,2.0,1.0,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
1,6300110000000000010115,594972,4.0,3.0,2.0,NaN,2.0,NaN,1.0,13,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
2,6300110000000000010115,594972,4.0,3.0,2.0,NaN,1.0,NaN,1.0,6,...,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,1.0
